# LAB03. Baseline разбора обращений
Модель ministral-14b-2512, Mistral API, 15.10.2026.
Анализ сохранённых ответов: 38 разных входов, 43 записи набора, 42 попытки.
Статьи заданы фикстурами. Полный протокол и команды запуска: ../docs/experiments/lab3.md.


In [1]:
import csv
import json
from pathlib import Path
root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'ml/experiments/evidence/baseline-source-manifest.json').is_file())
evidence = root / 'ml/experiments/evidence'
report = json.loads((evidence / 'baseline-report.json').read_text(encoding='utf-8'))
metric = report['independent_group_metrics']['free_api']
with (evidence / 'requests.csv').open(encoding='utf-8', newline='') as file:
    attempts = list(csv.DictReader(file))
result = {
    'dataset_records': report['coverage']['dataset_cases'],
    'distinct_inputs': metric['n'],
    'attempts': len(attempts),
    'initial_timeouts': sum(row['error'] == 'TimeoutError' for row in attempts),
    'schema': metric['schema_valid'],
    'mode': metric['mode_correct'],
    'category': metric['category_correct'],
    'false_answer_on_nonanswer_cases': metric['false_answer_on_nonanswer_cases'],
    'tokens': metric['tokens']['sum_observed_total'],
    'p50_ms': metric['latency']['p50_ms'],
    'p95_ms': metric['latency']['empirical_p95_ms'],
}
assert (result['dataset_records'], result['distinct_inputs'], result['attempts'], result['tokens']) == (43, 38, 42, 96484)
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "dataset_records": 43,
  "distinct_inputs": 38,
  "attempts": 42,
  "initial_timeouts": 4,
  "schema": {
    "numerator": 38,
    "denominator": 38,
    "value": 1.0
  },
  "mode": {
    "numerator": 33,
    "denominator": 38,
    "value": 0.868421052631579
  },
  "category": {
    "numerator": 36,
    "denominator": 38,
    "value": 0.9473684210526315
  },
  "false_answer_on_nonanswer_cases": {
    "numerator": 0,
    "denominator": 28,
    "value": 0.0
  },
  "tokens": 96484,
  "p50_ms": 5103.686,
  "p95_ms": 15505.612
}


Схема соблюдена в 38/38 ответах, режим выбран верно в 33/38, категория — в 36/38.
Четыре первоначальных тайм-аута сохранены в requests.csv.
Основная группа ошибок — уточнение вместо передачи специалисту.
Набор использовался при настройке; задержка включает API генерации, но не весь интерфейс и БД.
Для повторной проверки сохранённых данных: python ml/experiments/check_baseline.py из корня.
